« model_y — TINYSTORIES EĞİTİMİ (train_tinystories) · COLAB KOŞULARI »

**Soru: akrabalık verisinde ölçülen Model X, gerçek bir dilde (TinyStories) ne öğreniyor ve metni nasıl yazıyor?**
Model: Model X (`deneme2/model_y/model_y.py`; boyutlar KOŞU hücresinde `MODEL_KW` ile açık yazılır, varsayılanlar = Model X). Bu klasör yalnız TinyStories eğitimine özgü: veri `data_tinystories.py`, sınav `exam_tinystories.py`, çalıştırıcı `colab_tinystories.py`.
Karar kuralı ve sonuçlar: `train_tinystories/OLCULENLER_tinystories.md` (git dışı).

| hücre | iş |
|---|---|
| 0 HAZIRLIK | Drive, kod (GitHub'dan taze), veri (Drive önbelleği; iz yeniden hesaplanır, sabitle karşılaştırılır) |
| 1 KOŞU | GPU kapısı; eğitim arka planda (`colab_tinystories.start`); `STEPS` kullanıcı kararıyla yazılır |
| 2 NABIZ | durum ve son sınav satırları |
| 3 SONUÇ | bütün valid + sabit alt küme; istemlerin ve valid hikâyelerinin devamı (kural 12). Koşu sürerken son sınavın metinleri |
| X / Y / Z | durdur · Drive'daki kayıt · GPU durumu |

**Veri (kural 9): Colab üretmez.** model_18'in önbelleği `MyDrive/tinystories/onbellek/*_tam_n8000_v4.npy`: sözlük 8.004 (`<dolgu> <eos> <bilinmeyen> <nl>` + 8.000 kelime), train 531.080.303 token / 2.707.058 hikâye, valid 5.365.123 token / 27.542 hikâye (`belge/bulgu/model_18_veri.md` §7).
Her hikâye bir pencere: `<eos> hikâye <eos>`, sağdan `<dolgu>`, 512'ye sığmayan hikâye atılır, bölünmez (train %1,64). Validation hiç eğitilmez; train'de birebir geçen valid hikâyesi sınava girmez.
Mini-batch `data_tinystories.batches`: adımın fonksiyonu (epok başına tohumlu permütasyon), sürdürmede aynı parça gelir. Her 500 adımda yedek (model + Adam + adım).

In [ ]:
# 0 HAZIRLIK  |  CPU  |  tekrar: GUVENLI
# Cekirdek dustuyse ONCE bu hucre.  Kod Drive'dan (MyDrive/model_y/code; yerelde python sync_code.py), iz dogrulanir.
# Train akisi 1,06 GB: Drive'dan okuma + iz + sizinti taramasi (sure olculmedi), RAM ~2 GB.
import os, sys, shutil

# Canli kosu varken moduller yeniden yuklenirse kosu listesi sifirlanir.
if 'colab_tinystories' in sys.modules:
    _live = [n for n, r in sys.modules['colab_tinystories'].RUNS.items() if r['thread'].is_alive()]
    assert not _live, 'CEKIRDEK CANLI, kosu suruyor: %s -- HAZIRLIK gerekmiyor, 2 NABIZ' % _live

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/model_y'            # kosu klasorleri: ROOT/<NAME>/
TS_DIR = '/content/drive/MyDrive/tinystories'      # TinyStories verisi; onbellek/ burada
os.makedirs(ROOT, exist_ok=True)

# Kod Drive'dan, push gerekmez (kullanici, 28 Eylul): Colab diskine kopyalanir, kosu sirasinda Drive'da degisen dosya
# kosuyu etkilemez.  Iz listesi tutmazsa (esitleme yarim) durur.
SRC = '/content/model_y'
if os.path.isdir(SRC):
    shutil.rmtree(SRC)
shutil.copytree(ROOT + '/code', SRC)
TS = SRC + '/train_tinystories'                    # TinyStories egitimi: veri, sinav, calistirici
sys.path[:] = [p for p in sys.path if '/sekerai/' not in p and not p.startswith(SRC)]   # eski kolun yollari cikar
for _p in (TS, SRC):
    sys.path.insert(0, _p)
for _m in ('sync_code', 'model_y', 'model_y_transformer', 'train_y', 'data_tinystories', 'exam_tinystories',
           'colab_tinystories', 'model_20', 'model_20_transformer', 'train_20'):
    sys.modules.pop(_m, None)          # taze kod gercekten yuklensin
import sync_code
_man = sync_code.verify(SRC)
CODE = '%s  git %s%s  (esitleme %s)' % (_man['fingerprint'], _man['git_head'][:7],
                                         ' + commit edilmemis degisiklik' if _man['uncommitted'] else '', _man['time'])
print('kod   ' + CODE)

import torch
import model_y as M
import train_y as TR
import data_tinystories as DT
import exam_tinystories as ET
import colab_tinystories as C

# Veri Drive'dan; iz YENIDEN HESAPLANIP sabitle karsilastirilir.  Veri degisirse IZ BILEREK guncellenir.
IZ = '4fd319b93063'              # birlesik iz: ilk HAZIRLIK'ta basildi (27 Eylul, Colab; train 8b280f161b4a),
                                 # kullanici "Tiny stories başlayalım" ile sabitlendi
IZ_PARTS = {'vocab': 'c5af5d49cfab', 'valid': 'adaf94d61332'}   # yerelde, Drive dosyalarindan (27 Eylul)
DATA = DT.build(TS_DIR)
print(DT.audit(DATA))
assert all(DATA['fingerprints'][k] == v for k, v in IZ_PARTS.items()), 'sozluk/valid izi degisti: %s' % DATA['fingerprints']
assert IZ is None or DATA['fingerprint'] == IZ, 'veri izi %s, beklenen %s -- defter eski' % (DATA['fingerprint'], IZ)
if IZ is None:
    print('IZ SABITLENMEDI: %s -- IZ satirina yazilmadan KOSU baslamaz' % DATA['fingerprint'])

In [ ]:
# 1 KOSU -- Model Y (3 blok x 2 tur, 4 head, SwiGLU), lr "coherence" + WEIGHT_EMA, 10.000 adim  |  GPU  |
#     tekrar: RESUME=False iken ayni adla ikinci kez calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
SEED = 0                    # model baslangici VE batch sirasi (data_tinystories.batches)
NAME = 'tinystories_modely_3x2_s0'   # kullanici onayli, 28 Eylul
BATCH = 64
BUCKET = 32                 # uzunluga gore gruplama (kullanici, 27 Eylul: kabul)
_per = len(DATA['train_start']) // BATCH
STEPS = 10000               # kullanici, 28 Eylul: "Önce 10k adım"
SCHEDULE = 'coherence'      # lr = LR x ortalama(rho) ^ COHERENCE_POWER (train_y; ortalama pay/payda ayri)
FINAL_COOLDOWN = 0.05       # 10k kosularla ayni (son %5)
WEIGHT_EMA = 0.999
MODEL_KW = dict(d=384, layers=3, turns=6, heads=4, fact_activation='swiglu', units=1024, t_max=512)
# ayar A (kullanici, 28 Eylul: "bence yeni boyutla deneylim amacımız kıyas değli"); Model Y varsayilanlari: ogrenilen cikis
# olcegi, son tur alpha_F 1'den, bf16, LOSS_CHUNK 4096, Muon'da teget izdusum, coherence yansiz ortalama
EVERY = 500                 # sinav: sabit 1.000 valid hikayesi + ilk 4 istemin metni
SAVE_EVERY = 500            # kullanici, 27 Eylul: "yedek her 500 de olsun"
RESUME = False

# --- GPU KAPISI (CLAUDE.md kural 2); once biten kosularin bellegi birakilir
import gc, torch
gc.collect()
torch.cuda.empty_cache()
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_bos = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, 4 * BATCH * DATA['seq_len'] * len(DATA['vocab']) * 4 / 1e9)
assert _bos > _need, "GPU'da sadece %.1f GB bos, gereken ~%.1f GB" % (_bos, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB  gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _bos, _need))
assert IZ and DATA['fingerprint'] == IZ, 'veri izi sabitlenmedi ya da tutmuyor -- 0 HAZIRLIK'
assert (M.LEARN_OUTPUT_SCALE and M.LAST_FACTS_ALPHA_INIT == 1.0 and M.LOSS_CHUNK == 4096 and TR.MATMUL_PRECISION == 'bf16'
        and TR.MUON_TANGENT and TR.CANON and TR.NORMALIZED_UPDATE and TR.SPHERE_WEIGHTS), 'Model Y varsayilanlari degil -- 0 HAZIRLIK'

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme: 2 NABIZ
OUT = ROOT + '/' + NAME
C.start(NAME, DATA, OUT, steps=STEPS, seed=SEED, every=EVERY, device='cuda', setting='shared',
        batch_size=BATCH, bucket=BUCKET, model_kw=MODEL_KW, save_every=SAVE_EVERY, resume=RESUME, schedule=SCHEDULE,
        final_cooldown=FINAL_COOLDOWN, weight_ema=WEIGHT_EMA)
shutil.copyfile(SRC + '/fingerprints.json', OUT + '/fingerprints.json')   # hangi kod kostu
print('basladi: %s  kod %s  3 x 2, heads %d, FactUnits %s x %d, %s, son inis %%%d, weight_ema %s, %s  tohum %d  %d adim  ->  %s' % (
    NAME, CODE[:12], MODEL_KW['heads'], MODEL_KW['fact_activation'], MODEL_KW['units'], SCHEDULE,
    round(100 * FINAL_COOLDOWN), WEIGHT_EMA, TR.MATMUL_PRECISION, SEED, STEPS, OUT))

In [ ]:
# 2 NABIZ  |  CPU  |  tekrar: GUVENLI
C.pulse(5)

In [ ]:
# 3 SONUC  |  CPU  |  tekrar: GUVENLI
# SAYI + METIN (kural 12).  final.json yoksa (kosu suruyor) son sinavin sayilari ve metinleri.
import json
OUT = ROOT + '/' + NAME
_line = lambda v: 'n %d  nll %.4f  ppl %.2f  acc %.4f  bant %.3f/%.3f/%.3f  eos %.3f  ar %.3f  diger %.3f' % (
    v['n'], v['nll'], v['ppl'], v['accuracy'], v['acc_0_64'], v['acc_64_256'], v['acc_256_512'], v['eos_ok'],
    v['acc_ar'], v['acc_other'])
if os.path.exists(OUT + '/final.json'):
    _f = json.load(open(OUT + '/final.json', encoding='utf-8'))
    print('adim %d\nvalid   %s\nsubset  %s' % (_f['step'], _line(_f['valid']), _line(_f['subset'])))
    print('istemler: dongu %d/%d  farkli4 %.2f' % (_f['loops']['loop'], len(_f['prompts']), _f['loops']['distinct4']))
    for _w in _f['prompts']:
        print('\n== ISTEM\n%s\n-- model%s\n%s' % (_w['prompt'], '' if _w['ended'] else '  (<eos> yok, kesildi)', _w['model']))
    for _w in _f['stories']:
        print('\n== VALID HIKAYESI, ilk yari\n%s\n-- model\n%s\n-- gercek\n%s' % (_w['prompt'], _w['model'], _w['real']))
else:
    _e = json.load(open(OUT + '/exams.json', encoding='utf-8'))[-1]
    print('KOSU SURUYOR -- son sinav: adim %d  epok %.3f\nsubset  %s' % (_e['step'], _e['epoch'], _line(_e)))
    for _p, _t in zip(ET.PROMPTS, _e['texts']):
        print('\n== ISTEM\n%s\n-- model\n%s' % (_p, _t))

In [ ]:
# X DURDUR  |  CPU  |  tekrar: GUVENLI
# Bayrak koyar; iplik bir sonraki sinavda modeli Drive'a kaydedip cikar.
C.stop()

In [ ]:
# Y DRIVE'DAKI KAYIT  |  CPU  |  tekrar: GUVENLI
for _n in sorted(os.listdir(ROOT)):
    _d = ROOT + '/' + _n
    if os.path.isdir(_d) and _n.startswith('tinystories'):
        _fs = sorted(os.listdir(_d))
        _ck = [f for f in _fs if f.startswith('checkpoint_t')]
        print('%-34s %s  checkpoint %d (son %s)' % (_n, ' '.join(f for f in _fs if f not in _ck), len(_ck),
                                                   _ck[-1] if _ck else '-'))

In [ ]:
# Z GPU DURUMU  |  CPU  |  tekrar: GUVENLI
!nvidia-smi